# AlexNet on Colab GPU

Use this notebook with a GPU Colab kernel in VS Code. The notebook does not duplicate the model: it downloads the project and runs the existing Python scripts.

Before running code, choose a GPU runtime in Colab. For this private GitHub repository, create a read-only GitHub token and save it as a Colab Secret named `GITHUB_TOKEN`; never write the token directly in this notebook.

In [ ]:
import os
from getpass import getpass
import shutil
import subprocess
from pathlib import Path

# Paste the HTTPS URL of your GitHub repository here.
PROJECT_URL = "https://github.com/espstarry/pytorch-learning.git"
PROJECT_DIR = Path("/content/pytorch-learning")

try:
    from google.colab import userdata
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = os.environ.get("GITHUB_TOKEN")
if not github_token:
    github_token = getpass("GitHub token (hidden input, not saved): ")
print("GITHUB_TOKEN available:", bool(github_token))

if PROJECT_DIR.exists() and not (PROJECT_DIR / ".git").exists():
    shutil.rmtree(PROJECT_DIR)

if not PROJECT_DIR.exists():
    if not PROJECT_URL:
        raise ValueError("Set PROJECT_URL to your GitHub repository URL, then run this cell again.")
    clone_url = PROJECT_URL
    if github_token:
        clone_url = PROJECT_URL.replace("https://", f"https://x-access-token:{github_token}@", 1)
    result = subprocess.run(["git", "clone", clone_url, str(PROJECT_DIR)], text=True, capture_output=True)
    if result.returncode != 0:
        error = result.stderr.replace(github_token, "<hidden-token>")
        raise RuntimeError(f"Clone failed:\n{error}\nCheck the repository URL and token permissions, then run this cell again.")

os.chdir(PROJECT_DIR)
print("Project directory:", Path.cwd())

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU is attached. In Colab, change the runtime type to GPU.")

!nvidia-smi

## Smoke test

This runs one forward/backward/update step on a random image. It should finish quickly and print `device: cuda` when the GPU is active.

In [ ]:
os.chdir(PROJECT_DIR / "demos/vision/alexnet_110")
!python train.py

## CIFAR-10 training

First upload the CIFAR-10 archive from your computer. After extraction, the training code reads it from the local `data` folder.

The default command is a small learning run. Increase the sample counts and epochs only after it works.

In [ ]:
from google.colab import files
import tarfile
from pathlib import Path

uploaded = files.upload()
archive = Path(next(iter(uploaded)))
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with tarfile.open(archive, "r:gz") as tar:
    tar.extractall(data_dir)

archive.unlink()
print("CIFAR-10 extracted to:", data_dir.resolve())


In [ ]:
!pip install -q datasets -i https://pypi.tuna.tsinghua.edu.cn/simple


The training command below uses the Hugging Face CIFAR-10 dataset.


In [ ]:
# Small learning run: 1,000 training images, 200 test images, 1 epoch.
!python train_cifar10.py --source huggingface --epochs 1 --train-samples 1000 --test-samples 200

## Tensor export

This writes AlexNet intermediate tensors to `tensor_data.json`. Download the JSON after the cell finishes, then open the local Tensor Viewer and drag the file into it.

In [ ]:
!python export_tensors.py
from google.colab import files
files.download("tensor_data.json")